# 03c Transfer Evaluation


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
import pickle

model_dir = OUTPUT_DIR / 'models'
model_path = model_dir / 'berlin_ml_champion.pkl'
metadata_path = model_path.with_suffix('.pkl.metadata.json')

leipzig_finetune, leipzig_test = data_loading.load_leipzig_splits(DATA_DIR / 'phase_3_experiments')
feature_cols = data_loading.get_feature_columns(leipzig_test)

with (model_dir / 'berlin_scaler.pkl').open('rb') as handle:
    scaler = pickle.load(handle)

x_test_scaled = scaler.transform(leipzig_test[feature_cols].to_numpy(dtype=float))

model = training.load_model(model_path)
metadata = json.loads(metadata_path.read_text())
label_to_idx = {k: int(v) for k, v in metadata['label_to_idx'].items()}
idx_to_label = {int(k): v for k, v in metadata['idx_to_label'].items()}

y_test = leipzig_test['genus_latin'].map(label_to_idx).to_numpy()

preds = model.predict(x_test_scaled)
transfer_metrics = transfer.compute_transfer_metrics(
    y_test,
    preds,
    class_labels=list(idx_to_label.values()),
    include_ci=True,
)

serializable = dict(transfer_metrics)
if 'per_class' in serializable:
    serializable['per_class'] = serializable['per_class'].to_dict(orient='records')
if 'confusion_matrix' in serializable:
    serializable['confusion_matrix'] = np.array(serializable['confusion_matrix']).tolist()
if 'confidence_intervals' in serializable:
    serializable['confidence_intervals'] = {
        k: list(v) for k, v in serializable['confidence_intervals'].items()
    }

metrics_path = OUTPUT_DIR / 'metadata/transfer_evaluation.json'
metrics_path.write_text(json.dumps(serializable, indent=2))

fig_dir = OUTPUT_DIR / 'figures/transfer'
fig_dir.mkdir(parents=True, exist_ok=True)
cm = np.array(serializable['confusion_matrix'])
visualization.plot_confusion_matrix(cm, list(idx_to_label.values()), fig_dir / 'transfer_confusion_matrix.png')
